# Use Model for Predictions


In [10]:
from types import SimpleNamespace
import os
from IPython import get_ipython
import numpy as np
import torch
import lightning as pl

from data_module import NPZTensorDataModule
from model import MultiLayerLeakyReLUModel

detect if runnig as notebook or script


In [11]:
if get_ipython():
    is_running_in_notebook = True
    print("executing as notebook ...")
else:
    is_running_in_notebook = False
    print("executing as script ...")

executing as notebook ...


## Args

### Demo

In [12]:
# args = SimpleNamespace(
#     # io
#     data_path="data/demo",
#     logs_path="data/logs",
#     predicitions_path="data/predicted/demo",
#     experiment_name="demo_experiment",
#     version="v1",
#     # data
#     batch_size=64,
#     num_workers=4,
#     persistent_workers=True,
#     pin_memory=True,
# )

### laser-pulse-shaping-astra-sim

In [13]:
args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    predicitions_path="data/predicted/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    version="2025-12-13-13-34-11-ANimeJpN",
    # data
    batch_size=64,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

In [14]:
version_path = os.path.join(args.logs_path, args.experiment_name, args.version)

In [15]:
prediction_path = os.path.join(version_path, "predict")
os.makedirs(prediction_path, exist_ok=True)

In [16]:
ckpt_path = os.path.join(version_path, "best.ckpt")

## load data


In [17]:
data = NPZTensorDataModule(
    path=args.data_path,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    persistent_workers=args.persistent_workers,
    pin_memory=args.pin_memory,
)
data.setup("predict")

## load model


In [18]:
model = MultiLayerLeakyReLUModel.load_from_checkpoint(ckpt_path)

## Evaluate Best Model


### validation dataset


In [19]:
trainer = pl.Trainer(logger=False)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/dust/user/kwasniok/astra/opal-photo-injector-s ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


## predict


In [20]:
predictions = trainer.predict(model, datamodule=data)
predictions = torch.concat(predictions, dim=0)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Predicting: |          | 0/? [00:00<?, ?it/s]

### save


In [ ]:
out_path = os.path.join(args.predicitions_path, args.version, "predictions.npz")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
np.savez(out_path, y=predictions.numpy())